In [ ]:
import os
import pandas as pd
import ast

cell_types_polysome = ["HeLa", "HEK293T"]

## get the count data

In [ ]:
input_dir = "counts_in"
output_dir = "counts_out"

# Create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
# load the design file
design_df = pd.read_csv("all_lib2_designs.csv", index_col=0)

controls_dict = {
    "P96.2_3UTR": 1,
    "P96.4_3UTR": 1,
    "P96.5_3UTR": 0.1,
    "P96.6_3UTR": 0.1,
    "P96.7_3UTR": 0.01,
    "P96.8_3UTR": 0.01,
    "P96.9_3UTR": 0.001,
    "P96.10_3UTR": 0.001,
    "P96.11_3UTR": 0.0001,
    "P96.14_3UTR": 0.0001,
}
controls_names = list(controls_dict.keys())
unwanted_references = [key.replace("3UTR", "5UTR") for key in controls_dict.keys()]

In [ ]:
# counts should be given as a pandas dataframe with genes as columns and samples as rows
counts_df_actd = pd.read_csv(os.path.join(input_dir, "K562-ActD-lib2-counts.csv"), index_col=0)
counts_df_polysome = pd.read_csv(os.path.join(input_dir, "lib2-polysome-counts.csv"), index_col=0)

counts_df_actd.fillna(0, inplace=True)
counts_df_polysome.fillna(0, inplace=True)

In [ ]:
counts_df_actd.columns = [col.replace("-ActD-lib2", "") for col in counts_df_actd.columns]
counts_df_polysome.columns = [col.replace("-polysome-lib2", "") for col in counts_df_polysome.columns]
counts_df_polysome.columns = [col.replace("H293T", "HEK293T") for col in counts_df_polysome.columns]

In [ ]:
# which indices are in design_df but not in counts_df?
print("Indices in design_df but not in counts_df:")
print(set(design_df.index) - set(counts_df_actd.index))
# which indices are in counts_df but not in design_df?
print("Indices in counts_df but not in design_df:")
print(set(counts_df_actd.index) - set(design_df.index))

# which indices are in design_df but not in counts_df?
print("Indices in design_df but not in counts_df:")
print(set(design_df.index) - set(counts_df_polysome.index))
# which indices are in counts_df but not in design_df?
print("Indices in counts_df but not in design_df:")
print(set(counts_df_polysome.index) - set(design_df.index))

In [ ]:
# which entries have less than 10 reads at 0h for ActD?
actd_0h_filter = counts_df_actd.loc[counts_df_actd["K562_3UTR_0h"] < 10]
actd_0h_filter = actd_0h_filter[~actd_0h_filter.index.isin(controls_dict.keys())]
print(len(actd_0h_filter))

In [ ]:
# which entries have less than 50 reads total for polysome?
polysome_filter_list = []
for cell_type in cell_types_polysome:
    counts_df_polysome_cell_type = counts_df_polysome.loc[:, counts_df_polysome.columns.str.contains(cell_type)]
    # Get rows where sum of all columns except first is < 50
    polysome_filter = counts_df_polysome_cell_type[counts_df_polysome_cell_type.iloc[:, 1:].sum(axis=1) < 50]
    polysome_filter = polysome_filter[~polysome_filter.index.isin(controls_dict.keys())]
    print(len(polysome_filter))
    polysome_filter_list.append(polysome_filter)
polysome_filter = list(set(polysome_filter_list[0].index) | set(polysome_filter_list[1].index))
print(len(polysome_filter))

In [ ]:
# filter them
counts_df_polysome = counts_df_polysome[~counts_df_polysome.index.isin(polysome_filter)]
counts_df_actd = counts_df_actd[~counts_df_actd.index.isin(actd_0h_filter.index)]
# what is the length of the dataframes after filtering?
print(len(counts_df_polysome))
print(len(counts_df_actd))

## delete and transfer the controls in 5 UTRs



In [ ]:
counts_df_polysome = counts_df_polysome[~counts_df_polysome.index.isin(unwanted_references)]

for cell_type in cell_types_polysome:
    for column in counts_df_polysome.columns:
        if cell_type in column and "5UTR" in column:
            counts_df_polysome.loc[controls_names,column] = counts_df_polysome.loc[controls_names,column.replace("5UTR", "3UTR")]

## restore identical designs

In [ ]:
# get duplicated sequences
duplicated = pd.read_excel("lib2_duplicated_sequences.xlsx")

# make column 'dup' a list
duplicated['dup'] = duplicated['dup'].apply(ast.literal_eval)

In [ ]:
print("Length before duplicates (polysome): ", len(counts_df_polysome))
print("Length before duplicates (actd): ", len(counts_df_actd))

# iterate over duplicated sequences and add them to the counts
for index, row in duplicated.iterrows():
    seq1 = row['dup'][0]
    seq2 = row['dup'][1]
    # is seq1 in the counts?
    if seq1 in counts_df_polysome.index:
        counts_df_polysome.loc[seq2,:] = counts_df_polysome.loc[seq1,:]
    # is seq2 in the counts?
    elif seq2 in counts_df_polysome.index:
        counts_df_polysome.loc[seq1,:] = counts_df_polysome.loc[seq2,:]
    else:
        print(f"Neither {seq1} nor {seq2} are in the counts")

for index, row in duplicated.iterrows():
    seq1 = row['dup'][0]
    seq2 = row['dup'][1]
    # is seq1 in the counts?
    if seq1 in counts_df_actd.index:
        counts_df_actd.loc[seq2,:] = counts_df_actd.loc[seq1,:]
    # is seq2 in the counts?
    elif seq2 in counts_df_actd.index:
        counts_df_actd.loc[seq1,:] = counts_df_actd.loc[seq2,:]
    else:
        print(f"Neither {seq1} nor {seq2} are in the counts")

print("Length after duplicates (polysome): ", len(counts_df_polysome))
print("Length after duplicates (actd): ", len(counts_df_actd))


## Write out the results

In [ ]:
counts_df_polysome.to_csv(os.path.join(output_dir, "lib2-polysome-counts-processed.csv"))
counts_df_actd.to_csv(os.path.join(output_dir, "K562-ActD-lib2-counts-processed.csv"))